# Python进阶（四）：面向对象进阶
## 学习目标
- 理解封装、继承、多态三大特性
- 掌握抽象基类（ABC）的使用
- 理解并掌握魔法方法（__str__、__repr__、__add__等）
- 理解混入（Mixin）设计模式
- 了解元类（metaclass）的概念和应用

## 一、面向对象三大特性
### 1. 封装（Encapsulation）
隐藏对象的内部实现细节，只暴露必要的接口。

In [ ]:
"""封装示例"""
class Student:
    def __init__(self, name, score):
        self.name = name
        self.__score = score  # 私有属性
    
    def get_score(self):
        return self.__score
    
    def set_score(self, value):
        if 0 <= value <= 100:
            self.__score = value
        else:
            raise ValueError("成绩必须在0-100之间")

# 测试
s = Student("张三", 85)
print(s.get_score())  # 85
s.set_score(95)
print(s.get_score())  # 95

### 2. 继承（Inheritance）
子类继承父类的属性和方法，并可以重写或扩展。

In [ ]:
"""继承示例"""
class Animal:
    def speak(self):
        print("动物叫")

class Dog(Animal):  # Dog继承Animal
    def speak(self):  # 方法重写
        print("汪汪")

# 测试
a = Animal()
d = Dog()
a.speak()  # 动物叫
d.speak()  # 汪汪

### 3. 多态（Polymorphism）
同一个接口，不同的实现。

In [ ]:
"""多态示例"""
def make_it_speak(animal):
    animal.speak()  # 同一个接口，不同行为

class Cat(Animal):
    def speak(self):
        print("喵喵")

# 测试
make_it_speak(Dog())  # 汪汪
make_it_speak(Cat())  # 喵喵

## 二、抽象基类（ABC）
### 1. 为什么需要抽象基类？
- 定义通用规则和接口
- 强制子类实现特定方法
- 提高代码的规范性和一致性

In [ ]:
"""抽象基类示例"""
from abc import ABCMeta, abstractmethod

class Employee(metaclass=ABCMeta):
    """员工（抽象类）"""
    def __init__(self, name):
        self.name = name
    
    @abstractmethod
    def get_salary(self):
        """结算月薪（抽象方法）"""
        pass

class Manager(Employee):
    """部门经理"""
    def get_salary(self):
        return 15000.0

# Employee()  # 报错：不能实例化抽象类
m = Manager("曹操")
print(m.get_salary())  # 15000.0

### 2. 工厂模式（Factory Pattern）
通过工厂实现对象使用者和对象之间的解耦合。

In [ ]:
"""工厂模式示例"""
class EmployeeFactory:
    """创建员工的工厂"""
    @staticmethod
    def create(emp_type, *args, **kwargs):
        all_emp_types = {"M": Manager, "P": Programmer}
        cls = all_emp_types.get(emp_type.upper())
        return cls(*args, **kwargs) if cls else None

class Programmer(Employee):
    """程序员"""
    def __init__(self, name, working_hour=0):
        self.working_hour = working_hour
        super().__init__(name)
    
    def get_salary(self):
        return 200.0 * self.working_hour

# 使用工厂创建对象
emps = [
    EmployeeFactory.create("M", "曹操"),
    EmployeeFactory.create("P", "荀彧", 120)
]
for emp in emps:
    print(f"{emp.name}: {emp.get_salary():.2f}元")

## 三、魔法方法（Magic Methods）
### 1. 表示相关（让对象能被打印）
| 方法 | 触发时机 | 说明 |
|------|----------|------|
| `__str__(self)` | `print(obj)` / `str(obj)` | 面向用户的友好字符串 |
| `__repr__(self)` | `repr(obj)` / 交互式终端 | 面向开发者的精确描述 |

In [ ]:
"""魔法方法示例"""
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score
    
    def __str__(self):
        return f"学生：{self.name}，成绩：{self.score}"  # 给用户看
    
    def __repr__(self):
        return f"Student(name={self.name!r}, score={self.score})"  # 给开发者看

s = Student("张三", 95)
print(s)  # 调用 __str__
print(repr(s))  # 调用 __repr__

### 2. 运算符重载
| 方法 | 对应运算符 | 说明 |
|------|------------|------|
| `__add__(self, other)` | `a + b` | 加法 |
| `__eq__(self, other)` | `a == b` | 等于 |
| `__lt__(self, other)` | `a < b` | 小于 |

In [ ]:
"""运算符重载示例"""
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __add__(self, other):  # 重载 + 运算
        return Vector(self.x + other.x, self.y + other.y)
    
    def __eq__(self, other):  # 重载 == 运算
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)  # Vector(4, 6)
print(v1 == v2)  # False

### 3. 哈希与集合/字典键
- `__hash__(self)`：返回对象的哈希值
- `__eq__(self, other)`：判断两个对象是否相等
- **规则**：重写 `__eq__` 就必须重写 `__hash__`！

In [ ]:
"""哈希和字典键示例"""
class Card:
    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank
    
    def __hash__(self):
        return hash((self.suit, self.rank))
    
    def __eq__(self, other):
        return self.suit == other.suit and self.rank == other.rank
    
    def __repr__(self):
        return f"{self.suit}{self.rank}"

# 测试：放入set和dict
c1 = Card("♠", "A")
c2 = Card("♠", "A")
my_set = {c1, c2}
print(my_set)  # 只有一个元素（去重）

my_dict = {c1: "第一张"}
print(my_dict[c2])  # 第一张（能找到）

## 四、混入（Mixin）
Mixin是一种通过组合而非继承来复用代码的设计模式。

In [ ]:
"""Mixin示例"""
class SetOnceMappingMixin:
    """自定义混入类"""
    __slots__ = ()  # 节省内存
    
    def __setitem__(self, key, value):
        if key in self:
            raise KeyError(str(key) + " already set")
        return super().__setitem__(key, value)

class SetOnceDict(SetOnceMappingMixin, dict):
    """自定义字典"""
    pass

# 测试
my_dict = SetOnceDict()
my_dict["username"] = "jackfrued"
try:
    my_dict["username"] = "hellokitty"  # 报错
except KeyError as e:
    print(f"错误：{e}")

## 五、元类（Metaclass）
### 1. 核心概念
- 对象是类创建的，类是元类创建的
- 所有元类都直接或间接继承自 `type`
- 元类可以控制类的创建行为

### 2. 使用元类实现单例模式

In [ ]:
"""使用元类实现单例模式"""
import threading

class SingletonMeta(type):
    """自定义元类"""
    def __init__(cls, *args, **kwargs):
        cls.__instance = None
        cls.__lock = threading.RLock()
        super().__init__(*args, **kwargs)
    
    def __call__(cls, *args, **kwargs):
        if cls.__instance is None:
            with cls.__lock:
                if cls.__instance is None:
                    cls.__instance = super().__call__(*args, **kwargs)
        return cls.__instance

class President(metaclass=SingletonMeta):
    """总统（单例类）"""
    pass

# 测试
p1 = President()
p2 = President()
print(p1 is p2)  # True（同一个实例）

## 六、面向对象设计原则（SOLID）
1. **S** - 单一职责原则（SRP）
2. **O** - 开闭原则（OCP）
3. **L** - 里氏替换原则（LSP）
4. **I** - 接口隔离原则（ISP）
5. **D** - 依赖倒转原则（DIP）

## 七、练习与自测
### 练习题1：实现自定义字典
**题目**：实现一个自定义字典，限制只有在指定的key不存在时才能在字典中设置键值对。
**提示**：使用Mixin设计模式。

In [ ]:
"""练习题1解答"""
# 答案见上面的Mixin示例
my_dict = SetOnceDict()
my_dict["name"] = "Alice"
print(my_dict["name"])  # Alice
try:
    my_dict["name"] = "Bob"
except KeyError:
    print("Key already exists!")

### 练习题2：实现向量运算
**题目**：实现向量类，支持加法、减法、点积运算。
**提示**：重载运算符 `__add__`、`__sub__`、`__mul__`。

In [ ]:
"""练习题2解答"""
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)
    
    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)
    
    def __mul__(self, other):  # 点积
        return self.x * other.x + self.y * other.y
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

# 测试
v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)  # Vector(4, 6)
print(v1 - v2)  # Vector(-2, -2)
print(v1 * v2)  # 11

## 八、知识总结
### 重点记忆
1. **面向对象三大特性**
   - 封装：隐藏实现细节
   - 继承：代码复用
   - 多态：同一接口，不同实现
2. **抽象基类**
   - 使用 `@abstractmethod` 标记抽象方法
   - 不能实例化，必须被子类继承
3. **魔法方法**
   - `__str__` / `__repr__`：打印对象
   - `__add__` / `__eq__`：运算符重载
   - `__hash__` / `__eq__`：作为字典键/集合元素
4. **设计模式**
   - Mixin：代码复用
   - 工厂模式：解耦合
   - 单例模式：唯一实例
5. **元类**
   - 控制类的创建行为
   - 应用场景：ORM、注册机制、单例模式